# RL custom
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/v1/cookbook/notebooks/rl_custom.ipynb)

Define a charged-residue reward, run a short GRPO job, and compare scores before and after training.


## Setup
A GPU runtime is recommended: **Runtime → Change runtime type → GPU**. CPU execution is supported but slower.

Run cells in order. Installation is self-contained; no repository clone or account is needed. If Colab requests a session restart after installation, restart before running the imports. The first model load downloads weights.


In [ ]:
import sys

!"{sys.executable}" -m pip install -q "idiom @ git+https://github.com/rotskoff-group/idiom.git@v1"

In [ ]:
import gc
import time
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

from idiom import IDiom
from idiom.data.records import Record
from idiom.utils.notebook_helpers import save_run, train_and_save, training_config, write_fasta

started = time.perf_counter()

## Settings


In [ ]:
MODEL_ID = "jxliu2/idiom-300M"
DEVICE = "auto" # "cpu" or "cuda"; "auto" uses an available GPU
BATCH_SIZE = 1 # Prompts per training batch
SEED = 0
MAX_STEPS = 10 # Training steps for this demonstration
TARGET_LENGTH = 50 # Target IDR length in residues
MAX_NEW_TOKENS = 96 # Generation limit, including STOP; not a fixed IDR length
N = 8 # Sequences sampled from each model for comparison
OUT_DIR = Path("reward_design_outputs") / time.strftime("%Y%m%d-%H%M%S")
TARGET_CHARGED_FRACTION = 0.3 # Target fraction of D, E, K, and R residues

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Define a custom reward
This example targets the fraction of charged residues (D, E, K, R). Edit the scorer and `TARGET_CHARGED_FRACTION` together.

The factory returns a function with one finite score per input sequence, including empty sequences. Save it as a Python module so the training configuration can load it by path.


In [ ]:
from idiom.train.grpo.reward.resolve import load_callable

reward_file = OUT_DIR / "custom_reward.py"
reward_file.write_text('''def charged_fraction():
    def score(sequences):
        return [sum(s.count(a) for a in "DEKR") / max(len(s), 1) for s in sequences]
    return score
''')
reward_spec = dict(name=f"{reward_file.resolve()}:charged_fraction")
scorer = load_callable(reward_spec["name"])()
display(pd.DataFrame({"sequence": ["DEKR", "AAAA", ""], "score": scorer(["DEKR", "AAAA", ""])}))


## 2. Sample the starting model


In [ ]:
base = IDiom.from_pretrained(MODEL_ID, device=DEVICE)
sampling = dict(n=N, batch_size=BATCH_SIZE, seed=SEED, temperature=1.0, max_new_tokens=MAX_NEW_TOKENS)
baseline = base.generate_unprompted(**sampling)
write_fasta(
    [Record(f"baseline_{i}", s, 0, len(s)) for i, s in enumerate(baseline) if s],
    OUT_DIR / "baseline.fasta",
)
del base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 3. Train with GRPO
Combine the target reward with length and amino-acid entropy penalties, then run a short demonstration. Each quadratic term is `-((value - target) / (0.2 * target))²`. Training saves a reloadable model and local logs in `OUT_DIR`.


In [ ]:
cfg = training_config("grpo", MODEL_ID, OUT_DIR, device=DEVICE, steps=MAX_STEPS, seed=SEED)
cfg.prompts.n = max(16, MAX_STEPS * BATCH_SIZE)
cfg.prompts.batch_size = BATCH_SIZE
cfg.grpo.group_size = 2
cfg.grpo.max_new_tokens = MAX_NEW_TOKENS
cfg.grpo.lr = 5e-6
cfg.grpo.track_disorder = False
cfg.grpo.log_samples_every = 0
terms = [
    dict(label="length", weight=1.0, reward=dict(name="length"),
         shaping=dict(name="quadratic", target=TARGET_LENGTH, width=0.2)),
    dict(label="entropy", weight=1.0, reward=dict(name="entropy"),
         shaping=dict(name="quadratic", target=3.65, width=0.2)),
    dict(label="charged_fraction", weight=1.0, reward=reward_spec,
         shaping=dict(name="quadratic", target=TARGET_CHARGED_FRACTION, width=0.2)),
]

cfg.reward.terms = terms
release = train_and_save(cfg)

### Inspect training
Inspect the latest logged metrics. Full history is in `training/metrics/`. Blank entries indicate a metric was not logged on that step.


In [ ]:
metrics_path = next((OUT_DIR / "training/metrics").glob("version_*/metrics.csv"))
metrics = pd.read_csv(metrics_path)
columns = ["step"] + [name for name in ["train/loss", "train/reward", "train/kl"] if name in metrics]
display(metrics[columns].tail(3))

## 4. Generate from the trained model
Use the same sampling settings as the baseline for the comparison.


In [ ]:
adapted_model = IDiom.from_pretrained(release, device=DEVICE)
adapted = adapted_model.generate_unprompted(**sampling)
write_fasta(
    [Record(f"adapted_{i}", s, 0, len(s)) for i, s in enumerate(adapted) if s],
    OUT_DIR / "adapted.fasta",
)
del adapted_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 5. Compare before and after training
Compare raw scores and the total shaped reward. Short demonstration runs may not improve the objective and do not establish sequence function. Full per-sequence scores are saved in `rewards.csv`.


In [ ]:
from idiom.train.grpo.reward import build_reward

objective = build_reward(cfg.reward)
rows = []
for group, sequences in (("baseline", baseline), ("adapted", adapted)):
    totals, details = objective(sequences, group_size=1)
    scores = pd.DataFrame(details)
    scores["total_reward"] = totals
    scores["sequence"] = sequences
    scores["group"] = group
    rows.append(scores)
comparison = pd.concat(rows, ignore_index=True)
comparison.to_csv(OUT_DIR / "rewards.csv", index=False)
summary = comparison.groupby("group", sort=False).mean(numeric_only=True)
summary.to_csv(OUT_DIR / "reward_summary.csv")
score_columns = [f"{term['label']}_raw" for term in terms] + ["total_reward"]
display(summary[score_columns].round(3))
display(comparison[["group", "sequence", "total_reward"]].groupby("group", sort=False).head(2))

## Results
`baseline.fasta`, `adapted.fasta`, and `rewards.csv` contain the before/after samples and scores. `custom_reward.py` saves your scorer. `model/` is a reloadable release; `training/` contains logs and a checkpoint.

Download results from Colab’s Files pane before the runtime ends. `run.json` records settings and package versions.


In [ ]:
save_run(
    OUT_DIR,
    dict(model=MODEL_ID, device=DEVICE, seed=SEED, steps=MAX_STEPS, sampling=sampling),
    elapsed=time.perf_counter() - started,
)
print("Results folder:", OUT_DIR)